### Tools

In [1]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b"  # or "openai/gpt-oss-20b"
)

response = model.invoke("where is delhi")
response.content

'Delhi is the capital city and a Union Territory of India. It lies in the northern part of the country, on the banks of the Yamuna River, and is surrounded on three sides by the state of Haryana and on the southeast by the state of Uttar\u202fPradesh. Its geographic coordinates are approximately **28.61°\u202fN latitude and 77.23°\u202fE longitude**. In terms of distance, Delhi is about:\n\n- 250\u202fkm (≈155\u202fmi) north‑west of Agra  \n- 350\u202fkm (≈220\u202fmi) south‑west of Chandigarh  \n- 1,500\u202fkm (≈930\u202fmi) north‑west of Kolkata  \n- 2,150\u202fkm (≈1,340\u202fmi) south‑west of New\u202fDelhi’s nearest international airport, Indira Gandhi International (which is actually within Delhi itself).\n\nSo, if you look at a map of India, Delhi is situated in the north‑central region, just south of the Himalayan foothills.'

In [10]:
import os
from langchain.chat_models import init_chat_model


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-120b")

response=model.invoke("why do parrot talk?")
response.content

'Parrots don’t have a built‑in “language module” that lets them invent words the way humans do. What they do is **vocal mimicry**—they’re exceptionally good at copying sounds they hear in their environment. Several biological and behavioral factors make this possible:\n\n| Factor | How it helps a parrot “talk” |\n|--------|------------------------------|\n| **Highly developed syrinx** | The syrinx is the bird’s vocal organ (located at the base of the trachea). In parrots it’s especially flexible, with fine‑tuned muscles that let them produce a wide range of pitches, tones, and timbres—much more than most other birds. |\n| **Large brain region for vocal learning** | Parrots have an enlarged “song system” in the forebrain (the **nidopallium** and **arcopallium**). This area is analogous to the human Broca’s area and is specialized for hearing, memorizing, and reproducing complex sounds. |\n| **Social nature** | In the wild parrots live in flocks where constant vocal contact maintains gro

In [22]:
from langchain.tools import tool

@tool
def get_weather(city:str)->str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny"

model_with_tools=model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013594CFB360>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013594CFBCE0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather for a city.', 'parameters': {'properties': 

In [20]:

response=model_with_tools.invoke("whats the weather in the bengaluru?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Bengaluru. Use function get_weather.', 'tool_calls': [{'id': 'fc_772a4117-c84f-484f-9df7-b57a679a7a7c', 'function': {'arguments': '{"city":"Bengaluru"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 130, 'total_tokens': 174, 'completion_time': 0.092395656, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.011219411, 'prompt_tokens_details': None, 'queue_time': 0.443550991, 'total_time': 0.103615067}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a11c7c-e29b-78e3-bffe-dcbb65f1585b-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Bengaluru'}, 'id': 'fc_772a4117-c84f-484f-9df7-b57a679a7a7c', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_to

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

def get_weather(city:str)->str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny"

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

agent=create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are helpful assistant."
)

agent

### Tool Execution

In [24]:
#Step 1: Model generates the tool calls
messages=[{"role":"user","content":"what is the weather in Bengaluru"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and get results
for tool_call in ai_msg.tool_calls:
    #execute the tool with generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass the result back to model for final rsponse
final_response=model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Bengaluru is sunny.


In [25]:
messages

[{'role': 'user', 'content': 'what is the weather in Bengaluru'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks "what is the weather in Bengaluru". Need to fetch weather using function get_weather. Use function call.', 'tool_calls': [{'id': 'fc_e0a11247-45f6-4c7c-8db9-a396f074a6ed', 'function': {'arguments': '{"city":"Bengaluru"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 126, 'total_tokens': 180, 'completion_time': 0.114559364, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.006524332, 'prompt_tokens_details': None, 'queue_time': 0.338382382, 'total_time': 0.121083696}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6c36ac20de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11c88-0b11-7571-94b4-a93ba281cefc-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 